In [34]:
import pandas as pd
from pathlib import Path

# Set up local paths relative to notebook location
PROJECT_ROOT = Path("..").resolve()
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"


#Clean nav_history.csv — parse dates to datetime, sort by amfi_code + date, forward-fill missing NAV for holidays/weekends, remove duplicates, validate NAV > 0.

In [35]:
#Loading nav_history.csv
nav = pd.read_csv(
    f"{RAW_DATA}/02_nav_history.csv"
)
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [36]:
#Initial Inspection
print(f"Shape:\n{nav.shape}")
print("--"*50)
print(f"Information:")
nav.info()
print("--"*50)
print(f"Columns in the dataset: {nav.columns}")
print("--"*50)
print(f"Number of missing values:\n{nav.isnull().sum()}")
print("--"*50)
print(f"Number of duplicate rows: {nav.duplicated().sum()}")
print("--"*50)

Shape:
(46000, 3)
----------------------------------------------------------------------------------------------------
Information:
<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB
----------------------------------------------------------------------------------------------------
Columns in the dataset: Index(['amfi_code', 'date', 'nav'], dtype='str')
----------------------------------------------------------------------------------------------------
Number of missing values:
amfi_code    0
date         0
nav          0
dtype: int64
----------------------------------------------------------------------------------------------------
Number of duplicate rows: 0
-------------------------

In [37]:
#parse dates to datetime
nav['date']=pd.to_datetime(nav['date'])
print(f"Data types of the columns:\n{nav.dtypes}")

Data types of the columns:
amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object


In [38]:
#sort by amfi_code + date
nav = nav.sort_values(["amfi_code", "date"])
print(nav.sample())

       amfi_code       date      nav
29892     119093 2026-05-20  55.9676


In [39]:
#forward-fill missing NAV for holidays/weekends
print(f"Missing Values in Nav column:{nav['nav'].isnull().sum() }")
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

Missing Values in Nav column:0


In [40]:
#validate NAV > 0.
invalid_nav = nav[nav["nav"] <= 0]

print(len(invalid_nav))

0


In [41]:
#remove duplicates
print(f"Number of duplicate rows: {nav.duplicated().sum()}")
#nav = nav.drop_duplicates()

Number of duplicate rows: 0


In [42]:
#Exporting the cleaned nav history
nav.to_csv(f"{PROCESSED_DATA}/clean_nav_history.csv",index=False)
print(f"Done Exporting Cleaned NAV History")

Done Exporting Cleaned NAV History


#Clean investor_transactions.csv — standardise transaction_type values (SIP/Lumpsum/Redemption), validate amount > 0, fix date formats, check KYC status enum values.

In [43]:
#Load investor_transactions.csv
tx = pd.read_csv(
    f"{RAW_DATA}/08_investor_transactions.csv"
)
print(tx.head())


  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

In [44]:
#Initial Inspection
print(f"Shape:\n{tx.shape}")
print("--"*50)
print(f"Information:")
tx.info()
print("--"*50)
print(f"Columns in the dataset: {tx.columns}")
print("--"*50)
print(f"Number of missing values:\n{tx.isnull().sum()}")
print("--"*50)
print(f"Number of duplicate rows: {tx.duplicated().sum()}")
print("--"*50)

Shape:
(32778, 13)
----------------------------------------------------------------------------------------------------
Information:
<class 'pandas.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  str    
 1   transaction_date    32778 non-null  str    
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  str    
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  str    
 6   city                32778 non-null  str    
 7   city_tier           32778 non-null  str    
 8   age_group           32778 non-null  str    
 9   gender              32778 non-null  str    
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  str    
 12  kyc_status          32778 non-null  str    
dtypes: float64(1), int64(2), str(

In [45]:
#standardise transaction_type values (SIP/Lumpsum/Redemption)
print(f"Transaction Types :\n{tx["transaction_type"].value_counts(dropna=False)}")
#Standardize Transaction Types
tx["transaction_type"] = tx["transaction_type"].str.strip()
tx["transaction_type"].head()

Transaction Types :
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


0           SIP
1    Redemption
2           SIP
3           SIP
4       Lumpsum
Name: transaction_type, dtype: str

In [49]:
#validate amount > 0
invalid_amounts = tx[tx["amount_inr"] <= 0]
print(f"Invalid Amount :{invalid_amounts}")

Invalid Amount :Empty DataFrame
Columns: [investor_id, transaction_date, amfi_code, transaction_type, amount_inr, state, city, city_tier, age_group, gender, annual_income_lakh, payment_mode, kyc_status]
Index: []


In [47]:
#fix date formats
print(f"Datataype of the column before clean:\n{tx['transaction_date'].dtypes}")
tx["transaction_date"] = pd.to_datetime(tx["transaction_date"])
print(f"Datataype of the column after clean:\n{tx['transaction_date'].dtypes}")

Datataype of the column before clean:
str
Datataype of the column after clean:
datetime64[us]


In [50]:
#check KYC status enum values
tx["kyc_status"].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [51]:
#save the cleaned transactions.csv
tx.to_csv(f"{PROCESSED_DATA}/clean_transactions.csv",index=False)
print("Exported Successfully")

Exported Successfully


#Clean scheme_performance.csv — validate all return values are numeric, flag anomalies, check expense_ratio range (0.1% – 2.5%).

In [53]:
#load Clean scheme_performance.csv
perf = pd.read_csv(
    f"{RAW_DATA}/07_scheme_performance.csv"
)
print(perf.head(2))

   amfi_code                                scheme_name       fund_house  \
0     119551  SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552   SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0              11.49   0.87  0.89          0.88           1.29   
1               9.52   1.78  0.87          0.81           1.29   

   std_dev_ann_pct  max_drawdown_pct  aum_crore  expense_ratio_pct  \
0             14.0            -21.70      14288               1.54   
1             14.0            -24.43       1231               0.66   

   morningstar_rating risk_grade  
0                   4   Moderate  
1                   3   Moderate  


In [54]:
#Initial Inspection
print(f"Shape:\n{perf.shape}")
print("--"*50)
print(f"Information:")
perf.info()
print("--"*50)
print(f"Columns in the dataset: {perf}")
print("--"*50)
print(f"Number of missing values:\n{perf.isnull().sum()}")
print("--"*50)
print(f"Number of duplicate rows: {perf.duplicated().sum()}")
print("--"*50)

Shape:
(40, 19)
----------------------------------------------------------------------------------------------------
Information:
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     str    
 2   fund_house          40 non-null     str    
 3   category            40 non-null     str    
 4   plan                40 non-null     str    
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     fl

In [55]:
#validate all return values are numeric
numeric_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio"
]

for col in numeric_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

In [56]:
#checking for null values
perf[numeric_cols].isnull().sum()

return_1yr_pct    0
return_3yr_pct    0
return_5yr_pct    0
alpha             0
beta              0
sharpe_ratio      0
sortino_ratio     0
dtype: int64

In [59]:
#flag anomalies from negative_sharpe
negative_sharpe = perf[
    perf["sharpe_ratio"] < 0
]
print(f"negative_sharpe anomalies : {negative_sharpe}")

negative_sharpe anomalies : Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []


In [58]:
#check expense_ratio range (0.1% – 2.5%)
perf[
    (perf["expense_ratio_pct"] < 0.1)
    |
    (perf["expense_ratio_pct"] > 2.5)
]
print(perf['expense_ratio_pct'].sample(10))

26    1.38
21    1.56
4     0.77
20    1.59
30    0.79
24    1.64
16    0.72
36    1.60
8     0.78
17    1.53
Name: expense_ratio_pct, dtype: float64


In [60]:
#Save the cleaned scheme_performance
perf.to_csv(
    f"{PROCESSED_DATA}/clean_scheme_performance.csv",
    index=False
)
print("Cleaned CSV exported successfully")

Cleaned CSV exported successfully
